## Executive Summary

Slots Q, poles P and phases m do not fix a winding: the number of layers and the coil span are design choices. `design_windings(Q, P, m)` lists every winding the star of slots can produce for that machine that **can be wound into coils** and is **balanced**, best fundamental winding factor first, so the trade-offs are visible before you pick one.

| You give | You get |
|---|---|
| Q, P, m | All candidates: single layer and every double-layer coil span up to the pole pitch |
| … and `layers` | Only single-layer, or only double-layer, candidates |
| … and `w` | Double layer with that coil span |
| … `layers=2` and `w` | Exactly that winding |

Each candidate reports kw₁, the largest other MMF harmonic, and any sub-harmonics, and can produce its full winding-factor spectrum, MMF spectrum and MMF waveform.

If the filters leave nothing, `design_windings` raises `ValueError` with the reason for each rejected option, for example *"single layer: phase A has 2 go and 1 return conductors, so it can't be wound into coils"*.

## How It Fits Into Motor Design

```
03_sos.ipynb      layout engine: star of slots, coil matrix, check_winding
      │
      ▼
05_design.ipynb   ←── YOU ARE HERE: list and compare candidate windings
      │
      ▼
01_factors.ipynb / 02_mmf.ipynb   analyse the chosen winding
```

The other winding functions use the first candidate whenever `layers` or `w` is left out, so `winding_factor_spectrum(Q, P)` and `design_windings(Q, P)[0].winding_factors()` always describe the same winding.

## Rules

- **Candidates tried:** single layer, and double layer with coil spans w = 1 … ⌈Q/P⌉. Longer spans only repeat shorter ones mirrored about the pole pitch. An explicit `w` may be anything from 1 to Q − 1.
- **Kept only if** `check_winding` passes: each phase has as many go as return conductors, and the phase EMFs at ν = p are equal in magnitude and 360°/m apart.
- **Order:** highest kw₁ first; ties go to double layer, then to the shorter span (less end-winding copper).
- **`w` is the double-layer coil span.** Giving `w` without `layers` means double layer; `layers=1` with `w` is an error, since a single-layer winding has no free span.
- **Harmonics** are those of the balanced three-phase airgap MMF, normalised so the working harmonic ν = p is 1. "Largest other harmonic" excludes ν = p; sub-harmonics are the harmonics with ν < p.

**References:** Pyrhönen, Jokinen, Hrabovcová (2014), Ch. 2; Bianchi & Dai Pré, *IEE Proc. Electr. Power Appl.* 153(3), 2006.

## Imports and Setup

In [ ]:
#| default_exp winding.design

In [ ]:
#| export
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np

from emachines.winding.sos import _describe_option, _no_winding_error, _winding_options
from emachines.winding.winding_factors import winding_factor_spectrum
from emachines.winding.mmf import mmf_harmonics, mmf_waveform

---

## `WindingCandidate`

One buildable, balanced winding for a given (Q, P, m). It stores the layout and the summary numbers, and computes spectra and waveforms on request using the same functions as the rest of the module.

In [ ]:
#| export
@dataclass(frozen=True)
class WindingCandidate:
    """
    A buildable, balanced star-of-slots winding.

    Attributes
    ----------
    Q, P, m          : int          Slots, poles, phases
    layers           : int          1 (single layer) or 2 (double layer)
    w                : int | None   Double-layer coil span in slots (None for single layer)
    kw1              : float        Fundamental winding factor (at ν = p)
    largest_harmonic : tuple        (ν, amplitude) of the largest MMF harmonic other than
                                    ν = p, amplitude relative to the working harmonic;
                                    (None, 0.0) if there is none up to nu_max
    subharmonics     : tuple        MMF harmonic orders below p that are present
    nu_max           : int          Highest harmonic considered for the two fields above
    layout           : np.ndarray   (layers, Q) coil matrix, as from build_coil_matrix
    """
    Q: int
    P: int
    m: int
    layers: int
    w: int | None
    kw1: float
    largest_harmonic: tuple
    subharmonics: tuple
    nu_max: int
    layout: np.ndarray = field(repr=False, compare=False)

    @property
    def p(self) -> int:
        """Pole pairs."""
        return self.P // 2

    def describe(self) -> str:
        """One-line summary, e.g. 'double layer, w=5: kw1 = 0.9330, ...'."""
        nu, a = self.largest_harmonic
        other = f"largest other MMF harmonic ν={nu} at {a:.3f}" if nu else "no other MMF harmonics"
        sub = f"sub-harmonics ν={', '.join(map(str, self.subharmonics))}" if self.subharmonics else "no sub-harmonics"
        return f"{_describe_option(self.layers, self.w)}: kw1 = {self.kw1:.4f}, {other}, {sub}"

    def winding_factors(self, nu_max: int = 40) -> tuple[np.ndarray, np.ndarray]:
        """|kw(ν)| for ν = 1 … nu_max (see ``winding_factor_spectrum``)."""
        return winding_factor_spectrum(self.Q, self.P, self.m, self.layers, self.w, nu_max=nu_max)

    def mmf_harmonics(self, nu_max: int = 40, mode: str = "three_phase",
                      normalize: bool = True) -> tuple[np.ndarray, np.ndarray]:
        """MMF harmonic spectrum (see ``mmf_harmonics``)."""
        return mmf_harmonics(self.Q, self.P, self.m, self.layers, self.w,
                             mode=mode, nu_max=nu_max, normalize=normalize)

    def mmf_waveform(self, theta_electrical: float = 0.0) -> tuple[np.ndarray, np.ndarray]:
        """MMF staircase around the airgap (see ``mmf_waveform``)."""
        return mmf_waveform(self.Q, self.P, self.m, self.layers, self.w, theta_electrical)

---

## `design_windings`

In [ ]:
#| export
def design_windings(
    Q: int,
    P: int,
    m: int = 3,
    layers: int | None = None,
    w: int | None = None,
    nu_max: int | None = None,
) -> list[WindingCandidate]:
    """
    List the buildable, balanced windings for (Q, P, m), best kw1 first.

    Parameters
    ----------
    Q      : int          Number of stator slots
    P      : int          Number of poles (even, >= 2)
    m      : int          Number of phases (odd, default 3)
    layers : int | None   1 or 2 to keep only that layer count; None for both
    w      : int | None   Double-layer coil span in slots; implies layers=2
    nu_max : int | None   Highest MMF harmonic summarised per candidate.
                          None → max(40, 4p).

    Returns
    -------
    list[WindingCandidate]
        Never empty: if no option passes, ValueError is raised instead.

    Raises
    ------
    ValueError
        If no candidate is left, with the reason each option was rejected;
        if ``layers=1`` is combined with ``w``; or for invalid P, m, layers, w.
    """
    accepted, rejected = _winding_options(Q, P, m, layers, w)
    if not accepted:
        raise _no_winding_error(Q, P, m, layers, w, rejected)

    p = P // 2
    nu_top = nu_max if nu_max is not None else max(40, 4 * p)
    out = []
    for lyr, span, kw1, matrix in accepted:
        nu, amp = mmf_harmonics(Q, P, m, lyr, span, mode="three_phase", nu_max=nu_top)
        others = [(int(n), float(a)) for n, a in zip(nu, amp) if n != p and a > 1e-9]
        largest = max(others, key=lambda x: x[1]) if others else (None, 0.0)
        subs = tuple(n for n, _ in others if n < p)
        out.append(WindingCandidate(Q=Q, P=P, m=m, layers=lyr, w=span, kw1=kw1,
                                    largest_harmonic=largest, subharmonics=subs,
                                    nu_max=nu_top, layout=matrix))
    return out

In [ ]:
from emachines.winding.design import design_windings

# Examples
assert [round(c.kw1, 4) for c in design_windings(12, 10)] == [0.9659, 0.933, 0.483]
c = design_windings(24, 4, w=5)[0]
assert (c.layers, c.w, round(c.kw1, 4)) == (2, 5, 0.933)

### Example: candidates for common machines

In [ ]:
from emachines.winding.design import design_windings

for Q, P in [(12, 10), (9, 8), (24, 4), (36, 8), (15, 8)]:
    print(f"{Q}s/{P}p  (pole pitch {Q / P:.2f} slots)")
    for c in design_windings(Q, P):
        print("   ", c.describe())
    print()

### Example: filters and rejections

In [ ]:
from emachines.winding.design import design_windings

for kwargs in [dict(layers=2), dict(layers=1), dict(w=5), dict(layers=2, w=5)]:
    print(f"24s/4p {kwargs}:")
    for c in design_windings(24, 4, **kwargs):
        print("   ", c.describe())

for Q, P, kwargs in [(9, 8, dict(layers=1)), (9, 8, dict(layers=1, w=1)), (24, 4, dict(layers=2, w=12))]:
    try:
        design_windings(Q, P, **kwargs)
    except ValueError as e:
        print(f"\n{Q}s/{P}p {kwargs} → ValueError: {e}")

---

## Tests

In [ ]:
#| hide
import itertools, math
import numpy as np
from emachines.winding.design import design_windings, WindingCandidate
from emachines.winding.sos import build_coil_matrix, check_winding, winding_factor_sos, is_valid_combination
from emachines.winding.winding_factors import winding_factor_spectrum
from emachines.winding.mmf import mmf_harmonics, mmf_distribution

def _summary(cands):
    return [(c.layers, c.w, round(c.kw1, 4)) for c in cands]

# ── Reference candidate lists ─────────────────────────────────────────────────
assert _summary(design_windings(24, 4)) == [
    (2, 6, 0.9659), (1, None, 0.9659), (2, 5, 0.933), (2, 4, 0.8365),
    (2, 3, 0.683), (2, 2, 0.483), (2, 1, 0.25)], _summary(design_windings(24, 4))
assert _summary(design_windings(12, 10)) == [(1, None, 0.9659), (2, 1, 0.933), (2, 2, 0.483)]
assert _summary(design_windings(9, 8)) == [(2, 1, 0.9452), (2, 2, 0.3283)]
assert _summary(design_windings(36, 8))[:2] == [(2, 4, 0.9452), (2, 5, 0.9452)], "tie → shorter span first"
assert _summary(design_windings(15, 8)) == [(2, 2, 0.9514), (2, 1, 0.7109)]
print("✓ reference candidate lists")

# Harmonic summaries
c = design_windings(12, 10)[0]
assert c.subharmonics == (1,) and c.largest_harmonic[0] == 1 and math.isclose(c.largest_harmonic[1], 1.34, abs_tol=0.01)
c = design_windings(24, 4, w=5)[0]
# Chording to w=5 suppresses ν=10 (5th elec); the slot harmonic ν = Q - p = 22 is then the largest
assert c.subharmonics == () and c.largest_harmonic[0] == 22 and math.isclose(c.largest_harmonic[1], 0.091, abs_tol=1e-3)
nu, a = c.mmf_harmonics()
assert a[9] < 0.02 < design_windings(24, 4, w=6)[0].mmf_harmonics()[1][9], "w=5 suppresses ν=10"
print("✓ harmonic summaries")

# ── Filters ───────────────────────────────────────────────────────────────────
assert all(c.layers == 2 for c in design_windings(24, 4, layers=2))
assert _summary(design_windings(24, 4, layers=1)) == [(1, None, 0.9659)]
assert _summary(design_windings(24, 4, w=5)) == [(2, 5, 0.933)]
assert _summary(design_windings(24, 4, layers=2, w=5)) == [(2, 5, 0.933)]
for args, kwargs, msg in [((9, 8), dict(layers=1), "2 go and 1 return"),
                          ((9, 8), dict(layers=1, w=1), "double-layer windings only"),
                          ((24, 4), dict(layers=2, w=12), "zero"),
                          ((7, 4), {}, "No buildable, balanced winding"),
                          ((12, 9), {}, "even number of poles"),
                          ((12, 4), dict(m=2), "odd number of phases")]:
    try:
        design_windings(*args, **kwargs)
        raise AssertionError(f"{args} {kwargs} should raise")
    except ValueError as e:
        assert msg in str(e), (args, kwargs, str(e))
print("✓ filters and error messages")

# ── Consistency sweep ─────────────────────────────────────────────────────────
# Every candidate passes check_winding; its numbers match the analysis
# functions; and the defaults of those functions are the first candidate.
n_combos = n_cands = 0
for Q in range(3, 49, 3):
    for P in range(2, 25, 2):
        p = P // 2
        valid = is_valid_combination(Q, P)
        try:
            cands = design_windings(Q, P, nu_max=30)
        except ValueError:
            assert not valid, f"{Q}s/{P}p: is_valid_combination True but no candidate"
            continue
        assert valid, f"{Q}s/{P}p: candidates exist but is_valid_combination is False"
        n_combos += 1
        for c in cands:
            n_cands += 1
            r = check_winding(c.layout, P)
            assert r['buildable'] and r['balanced'], (Q, P, c.layers, c.w, r)
            assert math.isclose(c.kw1, winding_factor_sos(p, Q, P, layers=c.layers, w=c.w), abs_tol=1e-12)
        best = cands[0]
        assert np.array_equal(build_coil_matrix(Q, P), best.layout), f"{Q}s/{P}p build_coil_matrix default"
        _, kw = winding_factor_spectrum(Q, P, nu_max=30)
        assert np.allclose(kw, best.winding_factors(nu_max=30)[1]), f"{Q}s/{P}p winding_factor_spectrum default"
        _, a = mmf_harmonics(Q, P, nu_max=30)
        assert np.allclose(a, best.mmf_harmonics(nu_max=30)[1]), f"{Q}s/{P}p mmf_harmonics default"
        assert np.allclose(mmf_distribution(Q, P), mmf_distribution(Q, P, layers=best.layers, w=best.w))
print(f"✓ consistency sweep: {n_combos} combinations, {n_cands} candidates")

# ── Optimality: brute force over every balanced double-layer layout ──────────
# For each span, no balanced layout (any phase and direction per coil) has a
# higher kw1 than the star-of-slots candidate.
def _brute_best_kw1(Q, P, w, m=3):
    p = P // 2
    th = 2 * np.pi * np.arange(Q) / Q
    coil = np.exp(1j * p * th) - np.exp(1j * p * th[(np.arange(Q) + w) % Q])
    n = Q // m
    best = 0.0
    for g in set(itertools.permutations([0] * n + [1] * n + [2] * n)):
        g = np.array(g)
        if g[0] != 0:
            continue
        for s in itertools.product((1, -1), repeat=Q - 1):
            s = np.array((1,) + s)
            E = np.array([np.sum(s[g == k] * coil[g == k]) for k in range(m)])
            if abs(E[0]) < 1e-9 or not np.allclose(abs(E), abs(E[0])):
                continue
            if not np.allclose(abs(np.angle(E[1:] / E[:-1])), 2 * np.pi / 3, atol=1e-6):
                continue
            best = max(best, abs(E[0]) / (2 * n))
    return best

for Q, P, w in [(6, 2, 3), (6, 4, 1), (9, 8, 1), (9, 10, 1)]:
    sos_kw1 = design_windings(Q, P, layers=2, w=w)[0].kw1
    brute = _brute_best_kw1(Q, P, w)
    assert math.isclose(sos_kw1, brute, abs_tol=1e-9), (Q, P, w, sos_kw1, brute)
    print(f"pass  {Q}s/{P}p w={w}: star of slots kw1 = {sos_kw1:.4f} = best of all balanced layouts")

print()
print("✓ All design tests passed")